DATA SOURCE: https://apps.fs.usda.gov/fia/datamart/datamart.html

In [ ]:
import pandas as pd
import numpy as np

# for Vermont State (VT)
df = pd.read_csv('/Users/fiodarzianiutsich/Desktop/projekt/data_cleaning/VT_TREE.csv')

Based on the attached file (data_info.pdf), we select the variables of interest and filter the data. We are only interested in red spruce measured at breast height rather than at the root. We include only live, accurately measured trees and reject the rest to build a precise statistical model and avoid introducing noise.

In [16]:
df = df[df.DIAHTCD == 1]              # measured at breast height
df = df[df.STATUSCD == 1]             # live trees
df = df[df.HTCD == 1]                 # accurate measurement
df = df[df.ACTUALHT == df.HT]         # unbroken trees
red_spruce = df[df.SPCD == 97]        # code for red spurce - 97

red_spruce = red_spruce[['INVYR', 'UNITCD',  
                         'COUNTYCD', 'PLOT', 'TREE', 'DIA', 'HT']]

In [17]:
red_spruce = red_spruce.rename(columns = {
'INVYR': 'inventory_year', 'UNITCD': 'survey_unit',
'COUNTYCD': 'country', 'PLOT': 'plot_number',
'TREE': 'tree_number', 'DIA': 'diametr', 'HT': 'height'})

In [18]:
red_spruce.head()

,inventory_year,survey_unit,country,plot_number,tree_number,diametr,height
59152,2003,2,5,988,2,10.8,55.0
59224,2003,2,5,1159,7,5.4,34.0
59225,2003,2,5,1159,8,5.8,42.0
59227,2003,2,5,1159,10,5.4,39.0
59228,2003,2,5,1159,11,6.5,46.0


We want to verify whether we have all the necessary data.

In [19]:
red_spruce.isnull().sum()

inventory_year    0
survey_unit       0
country           0
plot_number       0
tree_number       0
diametr           0
height            0
dtype: int64

To avoid time-related issues in modeling, we use measurements from a single year. This allows us to compare modeling results across different years using samples taken from those specific years.

Year with the biggest sample:

In [20]:
red_spruce.groupby('inventory_year').size().idxmax()

np.int64(2022)

In [21]:
YEAR = 2022
red_spruce_per_year = red_spruce[red_spruce.inventory_year == YEAR]

For a single tree, we only need one measurement. It may happen that a tree was measured multiple times in the same year.

In [22]:
grouped_by_trees = red_spruce_per_year.groupby(['survey_unit', 'country', 
                                       'plot_number', 'tree_number'])

actual_measurement = red_spruce_per_year.loc[grouped_by_trees.inventory_year.idxmax()]

In [23]:
actual_measurement['height'] = actual_measurement['height']*0.3048 # feet to meters
actual_measurement['diametr'] = actual_measurement['diametr']*2.54 # inches to centemetrs

In [24]:
actual_measurement['log_height'] = np.log(actual_measurement.height)
actual_measurement['log_diametr'] = np.log(actual_measurement.diametr)
actual_measurement.head()

,inventory_year,survey_unit,country,plot_number,tree_number,diametr,height,log_height,log_diametr
185487,2022,2,5,204,1,18.796,11.5824,2.449487,2.933644
185513,2022,2,5,204,3,19.050,12.1920,2.500780,2.947067
185489,2022,2,5,204,4,28.702,12.8016,2.549570,3.356967
185471,2022,2,5,204,5,32.258,18.2880,2.906245,3.473766
185473,2022,2,5,204,7,30.226,18.5928,2.922774,3.408702


In [25]:
actual_measurement.to_csv('red_spruce_per_year.csv', index=False, encoding='utf-8-sig')